**Data Import and Basic Operation**

In [2]:
import pandas as pd
import requests
import time
df = pd.read_csv("/content/city_day.csv")
print(df.shape)
df.head()
df = df.rename(columns={"PM2.5": "PM2_5"})
df.info()

(25555, 16)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25555 entries, 0 to 25554
Data columns (total 16 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   City        25555 non-null  object 
 1   Date        25555 non-null  object 
 2   PM2_5       21300 non-null  float64
 3   PM10        14853 non-null  float64
 4   NO          21046 non-null  float64
 5   NO2         22396 non-null  float64
 6   NOx         21534 non-null  float64
 7   NH3         15770 non-null  float64
 8   CO          23556 non-null  float64
 9   SO2         22056 non-null  float64
 10  O3          22045 non-null  float64
 11  Benzene     20433 non-null  float64
 12  Toluene     18146 non-null  float64
 13  Xylene      9138 non-null   float64
 14  AQI         21304 non-null  float64
 15  AQI_Bucket  21304 non-null  object 
dtypes: float64(13), object(3)
memory usage: 3.1+ MB


**Data Engineering 1**

In [3]:
df = df.copy()

# Make sure we have a single, consistent Date column
if "date" in df.columns:
    df.rename(columns={"date": "Date"}, inplace=True)

df["Date"] = pd.to_datetime(df["Date"]).dt.date
df["City"] = df["City"].astype(str).str.strip()



In [93]:
#In this code block we geocode the cities to later find out their Geographical Information!!!
def geocode_city_openmeteo(city, country_code="IN"):
    url = "https://geocoding-api.open-meteo.com/v1/search"
    params = {"name": city, "count": 1, "language": "en", "format": "json", "country_code": country_code}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    if "results" not in js or len(js["results"]) == 0:
        return None
    best = js["results"][0]
    return {"City": city, "latitude": best["latitude"], "longitude": best["longitude"]}

cities = sorted(df["City"].unique())

geo_rows, failed_geo = [], []
for c in cities:
    g = geocode_city_openmeteo(c, country_code="IN")
    if g is None:
        failed_geo.append(c)
    else:
        geo_rows.append(g)
    time.sleep(0.3)

geo_df = pd.DataFrame(geo_rows)
print("Geocoded:", len(geo_df), "Failed:", len(failed_geo))
print("Failed:", failed_geo)

Geocoded: 21 Failed: 1
Failed: ['Brajrajnagar']


**Geographical Features Access**

In [5]:
session = requests.Session()

def fetch_openmeteo_daily_with_retry(lat, lon, start_date, end_date, timezone="Asia/Kolkata",
                                     max_retries=8, base_sleep=2.0):
    url = "https://archive-api.open-meteo.com/v1/archive"
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": start_date,
        "end_date": end_date,
        "daily": [
            "temperature_2m_max",
            "temperature_2m_min",
            "relative_humidity_2m_mean",
            "precipitation_sum",
            "wind_speed_10m_max"
        ],
        "timezone": timezone
    }

    for attempt in range(max_retries):
        r = session.get(url, params=params, timeout=60)

        if r.status_code == 200:
            js = r.json()
            w = pd.DataFrame(js["daily"]).rename(columns={"time": "Date"})
            w["Date"] = pd.to_datetime(w["Date"]).dt.date
            return w

        if r.status_code == 429:
            sleep_s = min(base_sleep * (2 ** attempt), 60)
            print(f"429 rate limit. Sleeping {sleep_s:.1f}s...")
            time.sleep(sleep_s)
            continue

        r.raise_for_status()

    raise RuntimeError("Failed after retries")

weather_all = []

for _, row in geo_df.iterrows():
    c = row["City"]
    lat, lon = row["latitude"], row["longitude"]

    df_c = df[df["City"] == c]
    start_date = str(df_c["Date"].min())
    end_date   = str(df_c["Date"].max())

    try:
        w = fetch_openmeteo_daily_with_retry(lat, lon, start_date, end_date)
        w["City"] = c
        weather_all.append(w)
        print("✅ Weather:", c)
    except Exception as e:
        print("❌ Weather failed:", c, "->", e)

    time.sleep(1.0)

weather_df = pd.concat(weather_all, ignore_index=True)
print("weather_df shape:", weather_df.shape)
weather_df.head()


✅ Weather: Ahmedabad
✅ Weather: Aizawl
✅ Weather: Amaravati
✅ Weather: Amritsar
✅ Weather: Bengaluru
✅ Weather: Bhopal
✅ Weather: Chandigarh
✅ Weather: Chennai
✅ Weather: Delhi
✅ Weather: Gurugram
✅ Weather: Guwahati
✅ Weather: Hyderabad
✅ Weather: Jaipur
✅ Weather: Jorapokhar
✅ Weather: Kolkata
429 rate limit. Sleeping 2.0s...
✅ Weather: Lucknow
✅ Weather: Mumbai
✅ Weather: Patna
✅ Weather: Shillong
✅ Weather: Talcher
✅ Weather: Thiruvananthapuram
weather_df shape: (24699, 7)


,Date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,City
0,2015-01-01,24.3,16.9,67,0.0,14.8,Ahmedabad
1,2015-01-02,23.6,17.3,65,0.0,15.6,Ahmedabad
2,2015-01-03,25.4,15.1,70,0.0,11.4,Ahmedabad
3,2015-01-04,26.3,14.5,64,0.0,13.6,Ahmedabad
4,2015-01-05,25.4,13.7,53,0.0,15.2,Ahmedabad


In [94]:
#The code below merges both the csv and the weather features together.
df2 = df.merge(weather_df, on=["City", "Date"], how="left")

weather_cols = [
    "temperature_2m_max","temperature_2m_min",
    "relative_humidity_2m_mean","precipitation_sum",
    "wind_speed_10m_max"
]

print("Missing weather ratio:")
print(df2[weather_cols].isna().mean())


Missing weather ratio:
temperature_2m_max           0.033496
temperature_2m_min           0.033496
relative_humidity_2m_mean    0.033496
precipitation_sum            0.033496
wind_speed_10m_max           0.033496
dtype: float64


In [7]:
weather_df.to_csv("weather_cache.csv", index=False)
df2.to_csv("aqi_with_weather.csv", index=False)
print("✅ Saved weather_cache.csv and aqi_with_weather.csv")

✅ Saved weather_cache.csv and aqi_with_weather.csv


In [8]:
df = df.copy()
df["date"] = pd.to_datetime(df["Date"]).dt.date

In [95]:
weather_all = []
failed_weather = []

for _, row in geo_df.iterrows():

    c = row["City"]
    lat, lon = row["latitude"], row["longitude"]

    df_c = df[df["City"] == c]

    start_date = str(df_c["Date"].min())
    end_date   = str(df_c["Date"].max())

    try:
        w = fetch_openmeteo_daily_with_retry(lat, lon, start_date, end_date)
        w["City"] = c
        weather_all.append(w)
        print("✅ Weather:", c)

    except Exception as e:
        failed_weather.append(c)
        print("❌ Weather failed:", c)

    time.sleep(1.0)   # important to avoid 429

weather_df = pd.concat(weather_all, ignore_index=True)

print("Weather failures:", failed_weather)


✅ Weather: Ahmedabad
✅ Weather: Aizawl
✅ Weather: Amaravati
✅ Weather: Amritsar
✅ Weather: Bengaluru
✅ Weather: Bhopal
✅ Weather: Chandigarh
✅ Weather: Chennai
✅ Weather: Delhi
✅ Weather: Gurugram
✅ Weather: Guwahati
✅ Weather: Hyderabad
✅ Weather: Jaipur
✅ Weather: Jorapokhar
✅ Weather: Kolkata
✅ Weather: Lucknow
✅ Weather: Mumbai
✅ Weather: Patna
✅ Weather: Shillong
✅ Weather: Talcher
✅ Weather: Thiruvananthapuram
Weather failures: []


In [96]:
df2 = df.merge(weather_df, on=["City", "Date"], how="left")

weather_cols = [
    "temperature_2m_max",
    "temperature_2m_min",
    "relative_humidity_2m_mean",
    "precipitation_sum",
    "wind_speed_10m_max"
]

print("Missing weather ratio:")
print(df2[weather_cols].isna().mean())


Missing weather ratio:
temperature_2m_max           0.033496
temperature_2m_min           0.033496
relative_humidity_2m_mean    0.033496
precipitation_sum            0.033496
wind_speed_10m_max           0.033496
dtype: float64


In [97]:
print(weather_df.head())
print(weather_df.dtypes)


         Date  temperature_2m_max  temperature_2m_min  \
0  2015-01-01                24.3                16.9   
1  2015-01-02                23.6                17.3   
2  2015-01-03                25.4                15.1   
3  2015-01-04                26.3                14.5   
4  2015-01-05                25.4                13.7   

   relative_humidity_2m_mean  precipitation_sum  wind_speed_10m_max       City  
0                         67                0.0                14.8  Ahmedabad  
1                         65                0.0                15.6  Ahmedabad  
2                         70                0.0                11.4  Ahmedabad  
3                         64                0.0                13.6  Ahmedabad  
4                         53                0.0                15.2  Ahmedabad  
Date                          object
temperature_2m_max           float64
temperature_2m_min           float64
relative_humidity_2m_mean      int64
precipitation_sum      

In [13]:
print(df.head())
print(df.dtypes)


        City        Date  PM2_5  PM10  NO    NO2    NOx  NH3     CO    SO2  \
0  Ahmedabad  2015-01-01    NaN   NaN NaN  18.22  17.15  NaN   0.92  27.64   
1  Ahmedabad  2015-01-02    NaN   NaN NaN  15.69  16.46  NaN   0.97  24.55   
2  Ahmedabad  2015-01-03    NaN   NaN NaN  19.30  29.70  NaN  17.40  29.07   
3  Ahmedabad  2015-01-04    NaN   NaN NaN  18.48  17.97  NaN   1.70  18.59   
4  Ahmedabad  2015-01-05    NaN   NaN NaN  21.42  37.76  NaN  22.10  39.33   

       O3  Benzene  Toluene  Xylene  AQI AQI_Bucket        date  
0  133.36     0.00     0.02    0.00  NaN        NaN  2015-01-01  
1   34.06     3.68     5.50    3.77  NaN        NaN  2015-01-02  
2   30.70     6.80    16.40    2.25  NaN        NaN  2015-01-03  
3   36.08     4.43    10.14    1.00  NaN        NaN  2015-01-04  
4   39.31     7.01    18.89    2.78  NaN        NaN  2015-01-05  
City           object
Date           object
PM2_5         float64
PM10          float64
NO            float64
NO2           float64
NOx

In [14]:
df2.info()
df2.isna().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25555 entries, 0 to 25554
Data columns (total 22 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   City                       25555 non-null  object 
 1   Date                       25555 non-null  object 
 2   PM2_5                      21300 non-null  float64
 3   PM10                       14853 non-null  float64
 4   NO                         21046 non-null  float64
 5   NO2                        22396 non-null  float64
 6   NOx                        21534 non-null  float64
 7   NH3                        15770 non-null  float64
 8   CO                         23556 non-null  float64
 9   SO2                        22056 non-null  float64
 10  O3                         22045 non-null  float64
 11  Benzene                    20433 non-null  float64
 12  Toluene                    18146 non-null  float64
 13  Xylene                     9138 non-null   flo

,0
City,0
Date,0
PM2_5,4255
PM10,10702
NO,4509
NO2,3159
NOx,4021
NH3,9785
CO,1999
SO2,3499


**Data Engineering 2**

In [99]:
#We drop columns which have less impact on AQI(and on top of that they have missing values(a lot))
df2.drop(columns=['Xylene'], inplace=True)

KeyError: "['Xylene'] not found in axis"

In [16]:
df2.isna().sum()

,0
City,0
Date,0
PM2_5,4255
PM10,10702
NO,4509
NO2,3159
NOx,4021
NH3,9785
CO,1999
SO2,3499


In [101]:
#In this code we use a method(in this case interpolation) for filling the missing values in the column

cols = ['PM2_5', 'PM10', 'NO', 'NO2',
        'CO', 'SO2', 'O3']

for col in cols:
    df2[col] = df2.groupby('City')[col].transform(
        lambda x: x.interpolate(method='linear')
    )

In [18]:
df2.isna().sum()

,0
City,0
Date,0
PM2_5,2422
PM10,9399
NO,3106
NO2,1180
NOx,4021
NH3,9785
CO,6
SO2,1132


In [19]:
df2.head()

,City,Date,PM2_5,PM10,NO,NO2,NOx,NH3,CO,SO2,...,Benzene,Toluene,AQI,AQI_Bucket,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max
0,Ahmedabad,2015-01-01,NaN,NaN,NaN,18.22,17.15,NaN,0.92,27.64,...,0.00,0.02,NaN,NaN,2015-01-01,24.3,16.9,67.0,0.0,14.8
1,Ahmedabad,2015-01-02,NaN,NaN,NaN,15.69,16.46,NaN,0.97,24.55,...,3.68,5.50,NaN,NaN,2015-01-02,23.6,17.3,65.0,0.0,15.6
2,Ahmedabad,2015-01-03,NaN,NaN,NaN,19.30,29.70,NaN,17.40,29.07,...,6.80,16.40,NaN,NaN,2015-01-03,25.4,15.1,70.0,0.0,11.4
3,Ahmedabad,2015-01-04,NaN,NaN,NaN,18.48,17.97,NaN,1.70,18.59,...,4.43,10.14,NaN,NaN,2015-01-04,26.3,14.5,64.0,0.0,13.6
4,Ahmedabad,2015-01-05,NaN,NaN,NaN,21.42,37.76,NaN,22.10,39.33,...,7.01,18.89,NaN,NaN,2015-01-05,25.4,13.7,53.0,0.0,15.2


In [20]:
df2 = df2.drop(columns = ['AQI_Bucket','Toluene', 'NH3'], errors='ignore')

In [21]:
df2.head()

,City,Date,PM2_5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max
0,Ahmedabad,2015-01-01,NaN,NaN,NaN,18.22,17.15,0.92,27.64,133.36,0.00,NaN,2015-01-01,24.3,16.9,67.0,0.0,14.8
1,Ahmedabad,2015-01-02,NaN,NaN,NaN,15.69,16.46,0.97,24.55,34.06,3.68,NaN,2015-01-02,23.6,17.3,65.0,0.0,15.6
2,Ahmedabad,2015-01-03,NaN,NaN,NaN,19.30,29.70,17.40,29.07,30.70,6.80,NaN,2015-01-03,25.4,15.1,70.0,0.0,11.4
3,Ahmedabad,2015-01-04,NaN,NaN,NaN,18.48,17.97,1.70,18.59,36.08,4.43,NaN,2015-01-04,26.3,14.5,64.0,0.0,13.6
4,Ahmedabad,2015-01-05,NaN,NaN,NaN,21.42,37.76,22.10,39.33,39.31,7.01,NaN,2015-01-05,25.4,13.7,53.0,0.0,15.2


In [102]:
#It converts the Date column to datetime objects
df2['Date'] = pd.to_datetime(df2['Date'])
df2 = df2.sort_values(['City','Date'])

In [23]:
df2.head()

,City,Date,PM2_5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max
0,Ahmedabad,2015-01-01,NaN,NaN,NaN,18.22,17.15,0.92,27.64,133.36,0.00,NaN,2015-01-01,24.3,16.9,67.0,0.0,14.8
1,Ahmedabad,2015-01-02,NaN,NaN,NaN,15.69,16.46,0.97,24.55,34.06,3.68,NaN,2015-01-02,23.6,17.3,65.0,0.0,15.6
2,Ahmedabad,2015-01-03,NaN,NaN,NaN,19.30,29.70,17.40,29.07,30.70,6.80,NaN,2015-01-03,25.4,15.1,70.0,0.0,11.4
3,Ahmedabad,2015-01-04,NaN,NaN,NaN,18.48,17.97,1.70,18.59,36.08,4.43,NaN,2015-01-04,26.3,14.5,64.0,0.0,13.6
4,Ahmedabad,2015-01-05,NaN,NaN,NaN,21.42,37.76,22.10,39.33,39.31,7.01,NaN,2015-01-05,25.4,13.7,53.0,0.0,15.2


In [24]:
#This block helps us in learning the trends.
df2 = df2.sort_values(['City', 'Date'])

df2['AQI_rolling7'] = (
    df2.groupby('City')['AQI']
    .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

df2['AQI_change'] = df2['AQI'] - df2['AQI_rolling7']


In [25]:
df2.head()

,City,Date,PM2_5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change
0,Ahmedabad,2015-01-01,NaN,NaN,NaN,18.22,17.15,0.92,27.64,133.36,0.00,NaN,2015-01-01,24.3,16.9,67.0,0.0,14.8,NaN,NaN
1,Ahmedabad,2015-01-02,NaN,NaN,NaN,15.69,16.46,0.97,24.55,34.06,3.68,NaN,2015-01-02,23.6,17.3,65.0,0.0,15.6,NaN,NaN
2,Ahmedabad,2015-01-03,NaN,NaN,NaN,19.30,29.70,17.40,29.07,30.70,6.80,NaN,2015-01-03,25.4,15.1,70.0,0.0,11.4,NaN,NaN
3,Ahmedabad,2015-01-04,NaN,NaN,NaN,18.48,17.97,1.70,18.59,36.08,4.43,NaN,2015-01-04,26.3,14.5,64.0,0.0,13.6,NaN,NaN
4,Ahmedabad,2015-01-05,NaN,NaN,NaN,21.42,37.76,22.10,39.33,39.31,7.01,NaN,2015-01-05,25.4,13.7,53.0,0.0,15.2,NaN,NaN


In [26]:
#In this we use one more method to fill the empty spaces.
weather_cols = [
    "temperature_2m_max","temperature_2m_min",
    "relative_humidity_2m_mean","precipitation_sum",
    "wind_speed_10m_max"
]

# forward/back fill per city, then global median as last fallback
df2[weather_cols] = (
    df2.groupby("City")[weather_cols]
       .transform(lambda x: x.ffill().bfill())
)

df2[weather_cols] = df2[weather_cols].fillna(df2[weather_cols].median(numeric_only=True))

print("Missing weather ratio after fill:")
print(df2[weather_cols].isna().mean())


Missing weather ratio after fill:
temperature_2m_max           0.0
temperature_2m_min           0.0
relative_humidity_2m_mean    0.0
precipitation_sum            0.0
wind_speed_10m_max           0.0
dtype: float64


In [27]:
df2 = df2.dropna(subset=['AQI'])

In [28]:
features = [
    'PM10','PM2_5','NO2','NO','SO2','CO','O3',
    'construction_intensity',
    'temperature_2m_max','temperature_2m_min',
    'relative_humidity_2m_mean','precipitation_sum',
    'wind_speed_10m_max'
]


In [29]:
df2.head()

,City,Date,PM2_5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change
28,Ahmedabad,2015-01-29,83.13,NaN,NaN,28.71,33.72,6.93,49.52,59.76,0.02,209.0,2015-01-29,25.5,11.8,49.0,0.0,12.7,209.000000,0.000000
29,Ahmedabad,2015-01-30,79.84,NaN,NaN,28.68,41.08,13.85,48.49,97.07,0.04,328.0,2015-01-30,26.7,13.2,45.0,0.0,16.2,268.500000,59.500000
30,Ahmedabad,2015-01-31,94.52,NaN,NaN,32.66,52.61,24.39,67.39,111.33,0.24,514.0,2015-01-31,27.7,13.3,46.0,0.0,12.5,350.333333,163.666667
31,Ahmedabad,2015-02-01,135.99,NaN,NaN,42.08,84.57,43.48,75.23,102.70,0.40,782.0,2015-02-01,29.1,12.7,54.0,0.0,10.2,458.250000,323.750000
32,Ahmedabad,2015-02-02,178.33,NaN,NaN,35.31,72.80,54.56,55.04,107.38,0.46,914.0,2015-02-02,28.7,13.6,61.0,0.0,9.1,549.400000,364.600000


In [30]:
for col in cols:
    df2[col] = df2.groupby('City')[col].transform(
        lambda x: x.ffill().bfill()
    )


In [31]:
df2[cols].isna().sum()

,0
PM2_5,0
PM10,1811
NO,1254
NO2,0
CO,0
SO2,0
O3,0


In [32]:
df2.head()

,City,Date,PM2_5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change
28,Ahmedabad,2015-01-29,83.13,122.41,NaN,28.71,33.72,6.93,49.52,59.76,0.02,209.0,2015-01-29,25.5,11.8,49.0,0.0,12.7,209.000000,0.000000
29,Ahmedabad,2015-01-30,79.84,122.41,NaN,28.68,41.08,13.85,48.49,97.07,0.04,328.0,2015-01-30,26.7,13.2,45.0,0.0,16.2,268.500000,59.500000
30,Ahmedabad,2015-01-31,94.52,122.41,NaN,32.66,52.61,24.39,67.39,111.33,0.24,514.0,2015-01-31,27.7,13.3,46.0,0.0,12.5,350.333333,163.666667
31,Ahmedabad,2015-02-01,135.99,122.41,NaN,42.08,84.57,43.48,75.23,102.70,0.40,782.0,2015-02-01,29.1,12.7,54.0,0.0,10.2,458.250000,323.750000
32,Ahmedabad,2015-02-02,178.33,122.41,NaN,35.31,72.80,54.56,55.04,107.38,0.46,914.0,2015-02-02,28.7,13.6,61.0,0.0,9.1,549.400000,364.600000


In [33]:
df2.head()

,City,Date,PM2_5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change
28,Ahmedabad,2015-01-29,83.13,122.41,NaN,28.71,33.72,6.93,49.52,59.76,0.02,209.0,2015-01-29,25.5,11.8,49.0,0.0,12.7,209.000000,0.000000
29,Ahmedabad,2015-01-30,79.84,122.41,NaN,28.68,41.08,13.85,48.49,97.07,0.04,328.0,2015-01-30,26.7,13.2,45.0,0.0,16.2,268.500000,59.500000
30,Ahmedabad,2015-01-31,94.52,122.41,NaN,32.66,52.61,24.39,67.39,111.33,0.24,514.0,2015-01-31,27.7,13.3,46.0,0.0,12.5,350.333333,163.666667
31,Ahmedabad,2015-02-01,135.99,122.41,NaN,42.08,84.57,43.48,75.23,102.70,0.40,782.0,2015-02-01,29.1,12.7,54.0,0.0,10.2,458.250000,323.750000
32,Ahmedabad,2015-02-02,178.33,122.41,NaN,35.31,72.80,54.56,55.04,107.38,0.46,914.0,2015-02-02,28.7,13.6,61.0,0.0,9.1,549.400000,364.600000


In [34]:
df2.isna().sum()

,0
City,0
Date,0
PM2_5,0
PM10,1811
NO,1254
NO2,0
NOx,1762
CO,0
SO2,0
O3,0


In [35]:
df2 = df2.dropna(subset=['AQI_rolling7', 'AQI_change'])

In [36]:
df2 = df2.copy()

In [37]:
for col in ['PM10', 'NO']:
    df2[col] = df2[col].fillna(df2[col].median())

In [38]:
df2.head()

,City,Date,PM2_5,PM10,NO,NO2,NOx,CO,SO2,O3,Benzene,AQI,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change
28,Ahmedabad,2015-01-29,83.13,122.41,10.05,28.71,33.72,6.93,49.52,59.76,0.02,209.0,2015-01-29,25.5,11.8,49.0,0.0,12.7,209.000000,0.000000
29,Ahmedabad,2015-01-30,79.84,122.41,10.05,28.68,41.08,13.85,48.49,97.07,0.04,328.0,2015-01-30,26.7,13.2,45.0,0.0,16.2,268.500000,59.500000
30,Ahmedabad,2015-01-31,94.52,122.41,10.05,32.66,52.61,24.39,67.39,111.33,0.24,514.0,2015-01-31,27.7,13.3,46.0,0.0,12.5,350.333333,163.666667
31,Ahmedabad,2015-02-01,135.99,122.41,10.05,42.08,84.57,43.48,75.23,102.70,0.40,782.0,2015-02-01,29.1,12.7,54.0,0.0,10.2,458.250000,323.750000
32,Ahmedabad,2015-02-02,178.33,122.41,10.05,35.31,72.80,54.56,55.04,107.38,0.46,914.0,2015-02-02,28.7,13.6,61.0,0.0,9.1,549.400000,364.600000


In [39]:
df2.isna().sum()

,0
City,0
Date,0
PM2_5,0
PM10,0
NO,0
NO2,0
NOx,1762
CO,0
SO2,0
O3,0


In [40]:
df2[['City','Date','AQI','AQI_rolling7','AQI_change']].head(20)


,City,Date,AQI,AQI_rolling7,AQI_change
28,Ahmedabad,2015-01-29,209.0,209.000000,0.000000
29,Ahmedabad,2015-01-30,328.0,268.500000,59.500000
30,Ahmedabad,2015-01-31,514.0,350.333333,163.666667
31,Ahmedabad,2015-02-01,782.0,458.250000,323.750000
32,Ahmedabad,2015-02-02,914.0,549.400000,364.600000
33,Ahmedabad,2015-02-03,660.0,567.833333,92.166667
34,Ahmedabad,2015-02-04,294.0,528.714286,-234.714286
35,Ahmedabad,2015-02-05,149.0,520.142857,-371.142857
36,Ahmedabad,2015-02-06,190.0,500.428571,-310.428571
37,Ahmedabad,2015-02-07,247.0,462.285714,-215.285714


In [41]:
df2[df2['City'] == 'Ahmedabad'][['Date','AQI','AQI_rolling7','AQI_change']].head(20)


,Date,AQI,AQI_rolling7,AQI_change
28,2015-01-29,209.0,209.000000,0.000000
29,2015-01-30,328.0,268.500000,59.500000
30,2015-01-31,514.0,350.333333,163.666667
31,2015-02-01,782.0,458.250000,323.750000
32,2015-02-02,914.0,549.400000,364.600000
33,2015-02-03,660.0,567.833333,92.166667
34,2015-02-04,294.0,528.714286,-234.714286
35,2015-02-05,149.0,520.142857,-371.142857
36,2015-02-06,190.0,500.428571,-310.428571
37,2015-02-07,247.0,462.285714,-215.285714


In [42]:
df2['AQI_change_next'] = (
    df2.groupby('City')['AQI_change'].shift(-1)
)


In [43]:
df2 = df2.dropna(subset=['AQI_change_next'])


In [44]:
df2.head()

,City,Date,PM2_5,PM10,NO,NO2,NOx,CO,SO2,O3,...,AQI,date,temperature_2m_max,temperature_2m_min,relative_humidity_2m_mean,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change,AQI_change_next
28,Ahmedabad,2015-01-29,83.13,122.41,10.05,28.71,33.72,6.93,49.52,59.76,...,209.0,2015-01-29,25.5,11.8,49.0,0.0,12.7,209.000000,0.000000,59.500000
29,Ahmedabad,2015-01-30,79.84,122.41,10.05,28.68,41.08,13.85,48.49,97.07,...,328.0,2015-01-30,26.7,13.2,45.0,0.0,16.2,268.500000,59.500000,163.666667
30,Ahmedabad,2015-01-31,94.52,122.41,10.05,32.66,52.61,24.39,67.39,111.33,...,514.0,2015-01-31,27.7,13.3,46.0,0.0,12.5,350.333333,163.666667,323.750000
31,Ahmedabad,2015-02-01,135.99,122.41,10.05,42.08,84.57,43.48,75.23,102.70,...,782.0,2015-02-01,29.1,12.7,54.0,0.0,10.2,458.250000,323.750000,364.600000
32,Ahmedabad,2015-02-02,178.33,122.41,10.05,35.31,72.80,54.56,55.04,107.38,...,914.0,2015-02-02,28.7,13.6,61.0,0.0,9.1,549.400000,364.600000,92.166667


In [45]:
for lag in [1, 3, 7]:
    df2[f'AQI_lag_{lag}'] = (
        df2.groupby('City')['AQI'].shift(lag)
    )


In [46]:
for lag in [1, 3]:
    df2[f'PM10_lag_{lag}'] = (
        df2.groupby('City')['PM10'].shift(lag)
    )


In [47]:
df2.head()

,City,Date,PM2_5,PM10,NO,NO2,NOx,CO,SO2,O3,...,precipitation_sum,wind_speed_10m_max,AQI_rolling7,AQI_change,AQI_change_next,AQI_lag_1,AQI_lag_3,AQI_lag_7,PM10_lag_1,PM10_lag_3
28,Ahmedabad,2015-01-29,83.13,122.41,10.05,28.71,33.72,6.93,49.52,59.76,...,0.0,12.7,209.000000,0.000000,59.500000,NaN,NaN,NaN,NaN,NaN
29,Ahmedabad,2015-01-30,79.84,122.41,10.05,28.68,41.08,13.85,48.49,97.07,...,0.0,16.2,268.500000,59.500000,163.666667,209.0,NaN,NaN,122.41,NaN
30,Ahmedabad,2015-01-31,94.52,122.41,10.05,32.66,52.61,24.39,67.39,111.33,...,0.0,12.5,350.333333,163.666667,323.750000,328.0,NaN,NaN,122.41,NaN
31,Ahmedabad,2015-02-01,135.99,122.41,10.05,42.08,84.57,43.48,75.23,102.70,...,0.0,10.2,458.250000,323.750000,364.600000,514.0,209.0,NaN,122.41,122.41
32,Ahmedabad,2015-02-02,178.33,122.41,10.05,35.31,72.80,54.56,55.04,107.38,...,0.0,9.1,549.400000,364.600000,92.166667,782.0,328.0,NaN,122.41,122.41


In [48]:
df2_sim = df2.copy()
df2_sim = df2_sim.sort_values(['City', 'Date']).reset_index(drop=True)

In [49]:
df2_sim['construction_intensity'] = 0.0

df2_sim['month'] = pd.to_datetime(df2_sim['Date']).dt.month
df2_sim['dow'] = pd.to_datetime(df2_sim['Date']).dt.dayofweek


In [104]:
#This code chooses a random value of construction intensity

import numpy as np

for city in df2_sim['City'].unique():

    city_mask = df2_sim['City'] == city
    city_dates = pd.to_datetime(df2_sim.loc[city_mask, 'Date'])

    for year in city_dates.dt.year.unique():

        year_mask = city_mask & (city_dates.dt.year == year)
        days = df2_sim.loc[year_mask, 'Date'].values

        if len(days) < 120:
            continue

        n_phases = np.random.choice([1,2,3], p=[0.2,0.5,0.3])

        for _ in range(n_phases):

            max_start = len(days) - 60
            if max_start <= 0:
                continue

            start_idx = np.random.randint(0, max_start)
            phase_len = np.random.randint(60, min(121, len(days)-start_idx))

            phase_dates = days[start_idx : start_idx + phase_len]

            mask = (df2_sim['Date'].isin(phase_dates)) & city_mask

            df2_sim.loc[mask, 'construction_intensity'] = \
                np.random.uniform(0.6, 1.0, size=mask.sum())


In [51]:
# Season boost (dry months higher)
df2_sim.loc[df2_sim['month'].isin([10,11,12,1,2,3,4,5]),
           'construction_intensity'] *= 1.15

# Monsoon suppression
df2_sim.loc[df2_sim['month'].isin([6,7,8,9]),
           'construction_intensity'] *= 0.7

# Weekend suppression
df2_sim.loc[df2_sim['dow'] >= 5,
           'construction_intensity'] *= 0.6

df2_sim['construction_intensity'] = df2_sim['construction_intensity'].clip(0,1)


In [52]:
df2_sim['construction_intensity'].mean()


np.float64(0.2564479746223581)

In [105]:
#In this block we add weights to the values, that would get affected the most due to construction
df2_sim['PM10'] = df2_sim['PM10'] * (1 + 0.45 * df2_sim['construction_intensity'])
df2_sim['PM2_5'] = df2_sim['PM2_5'] * (1 + 0.25 * df2_sim['construction_intensity'])
df2_sim['NO2'] = df2_sim['NO2'] * (1 + 0.07 * df2_sim['construction_intensity'])


**Feature Engineering**

In [54]:
df2_sim['construction_PM10'] = df2_sim['construction_intensity'] * df2_sim['PM10']
df2_sim['construction_PM25'] = df2_sim['construction_intensity'] * df2_sim['PM2_5']


In [55]:
df2_sim['AQI'] = (
    0.4 * df2_sim['PM2_5'] +
    0.3 * df2_sim['PM10'] +
    0.1 * df2_sim['NO2'] +
    0.1 * df2_sim['SO2'] +
    0.1 * df2_sim['O3']
)


In [56]:
df2_sim['construction_intensity'].describe()

,construction_intensity
count,21282.000000
mean,0.256448
std,0.349199
min,0.000000
25%,0.000000
50%,0.000000
75%,0.545813
max,1.000000


In [58]:
df2_sim['AQI_change_next'] = \
    df2_sim.groupby('City')['AQI'].shift(-1) - df2_sim['AQI']


In [59]:
df2_sim = df2_sim.dropna()


In [60]:
df2_sim['construction_rolling_7'] = (
    df2_sim.groupby('City')['construction_intensity']
          .rolling(7)
          .mean()
          .reset_index(level=0, drop=True)
)

df2_sim['construction_PM10'] = df2_sim['construction_intensity'] * df2_sim['PM10']


In [61]:
features = [
    'PM10','PM2_5','NO2','NO','SO2','CO','O3',
    'construction_intensity',
    'temperature_2m_max','temperature_2m_min',
    'relative_humidity_2m_mean','precipitation_sum',
    'wind_speed_10m_max'
]


X = df2_sim[features]
y = df2_sim['AQI_change_next']


In [108]:
df_feat = df2.copy()

# Ensure correct types
df_feat["Date"] = pd.to_datetime(df_feat["Date"])
df_feat = df_feat.sort_values(["City", "Date"])

# --- Lag features (AQI memory) ---
# Lag features help prediction because they store AQI values from previous days.

for lag in [1, 3, 7]:
    df_feat[f"AQI_lag_{lag}"] = df_feat.groupby("City")["AQI"].shift(lag)

# --- Rolling features (trend/volatility) ---
df_feat["AQI_roll7_mean"] = (
    df_feat.groupby("City")["AQI"]
    .rolling(window=7, min_periods=7)
    .mean()
    .reset_index(level=0, drop=True)
)

df_feat["AQI_roll7_std"] = (
    df_feat.groupby("City")["AQI"]
    .rolling(window=7, min_periods=7)
    .std()
    .reset_index(level=0, drop=True)
)

# Optional: rolling change (momentum)
df_feat["AQI_change_1d"] = df_feat.groupby("City")["AQI"].diff(1)
df_feat["AQI_change_3d"] = df_feat.groupby("City")["AQI"].diff(3)


In [110]:
np.random.seed(42)

if "construction_intensity" not in df_feat.columns:
    df_feat["construction_intensity"] = np.random.beta(2, 5, len(df_feat)) #Here we use beta distribution to simulte the values.

In [64]:
temporal_cols = [
    "AQI_lag_1","AQI_lag_3","AQI_lag_7",
    "AQI_roll7_mean","AQI_roll7_std",
    "AQI_change_1d","AQI_change_3d"
]

df_feat = df_feat.dropna(subset=temporal_cols).copy()
print("After temporal features, rows:", len(df_feat))

After temporal features, rows: 21128


In [65]:
features_updated = features + temporal_cols

In [66]:
df_sim = df_feat.sort_values(['City','Date'])

for lag in [1, 3, 7]:
    df_sim[f'AQI_lag_{lag}'] = df_sim.groupby('City')['AQI'].shift(lag)

df_sim['AQI_roll7_mean'] = df_sim.groupby('City')['AQI'].rolling(7).mean().reset_index(level=0, drop=True)
df_sim['AQI_roll7_std'] = df_sim.groupby('City')['AQI'].rolling(7).std().reset_index(level=0, drop=True)

In [67]:
print("Has construction_intensity?", "construction_intensity" in df_feat.columns)
print(df_feat.columns)


Has construction_intensity? True
Index(['City', 'Date', 'PM2_5', 'PM10', 'NO', 'NO2', 'NOx', 'CO', 'SO2', 'O3',
       'Benzene', 'AQI', 'date', 'temperature_2m_max', 'temperature_2m_min',
       'relative_humidity_2m_mean', 'precipitation_sum', 'wind_speed_10m_max',
       'AQI_rolling7', 'AQI_change', 'AQI_change_next', 'AQI_lag_1',
       'AQI_lag_3', 'AQI_lag_7', 'PM10_lag_1', 'PM10_lag_3', 'AQI_roll7_mean',
       'AQI_roll7_std', 'AQI_change_1d', 'AQI_change_3d',
       'construction_intensity'],
      dtype='object')


In [68]:
target = "AQI_change_next"

model_df = df_sim[["City", "Date"] + features_updated + [target]].copy()

# Drop rows where ANY feature/target is missing
model_df = model_df.dropna(subset=features_updated + [target]).reset_index(drop=True)

print("Model df shape:", model_df.shape)


Model df shape: (20974, 23)


In [69]:
print(df_sim.columns)

Index(['City', 'Date', 'PM2_5', 'PM10', 'NO', 'NO2', 'NOx', 'CO', 'SO2', 'O3',
       'Benzene', 'AQI', 'date', 'temperature_2m_max', 'temperature_2m_min',
       'relative_humidity_2m_mean', 'precipitation_sum', 'wind_speed_10m_max',
       'AQI_rolling7', 'AQI_change', 'AQI_change_next', 'AQI_lag_1',
       'AQI_lag_3', 'AQI_lag_7', 'PM10_lag_1', 'PM10_lag_3', 'AQI_roll7_mean',
       'AQI_roll7_std', 'AQI_change_1d', 'AQI_change_3d',
       'construction_intensity'],
      dtype='object')


In [70]:
backend_df = df_sim[[
    "City",
    "Date",
    "AQI",
    "PM10",
    "PM2_5",
    "NO2",
    "NO",
    "SO2",
    "CO",
    "O3"
]].copy()

backend_df = backend_df.sort_values(["City", "Date"])
backend_df.head()


,City,Date,AQI,PM10,PM2_5,NO2,NO,SO2,CO,O3
35,Ahmedabad,2015-02-05,149.0,122.41,58.36,21.39,10.05,32.66,2.60,53.54
36,Ahmedabad,2015-02-06,190.0,122.41,79.29,26.94,10.05,67.41,1.16,59.30
37,Ahmedabad,2015-02-07,247.0,122.41,88.70,31.32,10.05,80.09,7.29,44.76
38,Ahmedabad,2015-02-08,379.0,122.41,74.28,27.30,10.05,54.28,8.92,47.42
39,Ahmedabad,2015-02-09,341.0,122.41,113.93,24.27,10.05,48.73,4.32,39.94


In [71]:
backend_df.to_csv("city_day.csv", index=False)

In [72]:
from sklearn.model_selection import train_test_split
model_df = model_df.sort_values(["City", "Date"]).reset_index(drop=True)

split_index = int(len(model_df) * 0.8)
train = model_df.iloc[:split_index]
test  = model_df.iloc[split_index:]

X_train = train[features_updated]
y_train = train[target]

X_test = test[features_updated]
y_test = test[target]

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)


X_train: (16779, 20) y_train: (16779,)
X_test: (4195, 20) y_test: (4195,)


In [73]:
import lightgbm as lgb

reg = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42
)

reg.fit(X_train, y_train)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004049 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4905
[LightGBM] [Info] Number of data points in the train set: 16779, number of used features: 20
[LightGBM] [Info] Start training from score -0.248935


LGBMRegressor(learning_rate=0.05, n_estimators=300, random_state=42)

In [75]:
# ─── MODEL A: WEATHER-ONLY BASELINE ───────────────────────────
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
weather_features = [
    'temperature_2m_max', 'temperature_2m_min',
    'relative_humidity_2m_mean', 'precipitation_sum',
    'wind_speed_10m_max',
    # temporal AQI lags — these are fair game for both models
    'AQI_lag_1', 'AQI_lag_3', 'AQI_lag_7',
    'AQI_roll7_mean', 'AQI_roll7_std',
    'AQI_change_1d', 'AQI_change_3d'
]

X_train_A = train[weather_features]
X_test_A  = test[weather_features]

reg_A = lgb.LGBMRegressor(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=-1,
    random_state=42
)
reg_A.fit(X_train_A, y_train)
y_pred_A = reg_A.predict(X_test_A)

rmse_A = np.sqrt(mean_squared_error(y_test, y_pred_A))
mae_A  = mean_absolute_error(y_test, y_pred_A)
r2_A   = r2_score(y_test, y_pred_A)

print("=== MODEL A (Weather Only) ===")
print(f"RMSE: {rmse_A:.4f}")
print(f"MAE:  {mae_A:.4f}")
print(f"R²:   {r2_A:.4f}")

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002611 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2865
[LightGBM] [Info] Number of data points in the train set: 16779, number of used features: 12
[LightGBM] [Info] Start training from score -0.248935
=== MODEL A (Weather Only) ===
RMSE: 31.9439
MAE:  20.7826
R²:   0.3533


In [76]:
y_pred = reg.predict(X_test)

In [77]:
pred_df = pd.DataFrame({
    "Actual_AQI_Change": y_test.values,
    "Predicted_AQI_Change": y_pred
})

pred_df.head(10)


,Actual_AQI_Change,Predicted_AQI_Change
0,36.142857,65.037982
1,-119.000000,-31.468809
2,-7.428571,-80.205464
3,81.857143,17.478945
4,26.428571,54.432843
5,-21.428571,28.367206
6,14.000000,0.426956
7,61.714286,7.680910
8,-3.000000,21.914734
9,-25.714286,-36.076747


In [78]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)


RMSE: 29.533622591683535
MAE: 19.40210439342815
R2: 0.4472288493021348


In [79]:
sample_df = X_test.iloc[[0]]

print("Input features:")
print(sample_df)

pred_one = reg.predict(sample_df)[0]

print("\nPredicted AQI Change:", pred_one)


Input features:
         PM10   PM2_5    NO2     NO    SO2    CO     O3  \
16779  108.58  247.31  57.64  49.66  11.44  0.95  21.44   

       construction_intensity  temperature_2m_max  temperature_2m_min  \
16779                0.071329                21.9                 8.4   

       relative_humidity_2m_mean  precipitation_sum  wind_speed_10m_max  \
16779                       74.0                0.0                 9.7   

       AQI_lag_1  AQI_lag_3  AQI_lag_7  AQI_roll7_mean  AQI_roll7_std  \
16779      346.0      270.0      383.0      296.428571      63.274158   

       AQI_change_1d  AQI_change_3d  
16779           50.0          126.0  

Predicted AQI Change: 65.03798194055754


In [80]:
custom_dict = {
    # pollutants
    "PM10": 150,
    "PM2_5": 80,
    "NO2": 40,
    "NO": 30,
    "SO2": 20,
    "CO": 1.2,
    "O3": 35,

    # construction + weather
    "construction_intensity": 0.5,
    "temperature_2m_max": 32,
    "temperature_2m_min": 20,
    "relative_humidity_2m_mean": 65,
    "precipitation_sum": 0.0,
    "wind_speed_10m_max": 10,

    # temporal AQI features (must be provided!)
    "AQI_lag_1": 180,
    "AQI_lag_3": 165,
    "AQI_lag_7": 150,
    "AQI_roll7_mean": 158,
    "AQI_roll7_std": 22,
    "AQI_change_1d": 15,
    "AQI_change_3d": 30
}

custom_df = pd.DataFrame([custom_dict])[features_updated]
pred_custom = reg.predict(custom_df)[0]
print("Predicted AQI Change:", pred_custom)


Predicted AQI Change: 27.562485617328946


In [81]:
threshold = train["AQI_change_next"].quantile(0.90)

model_df["spike"] = (model_df["AQI_change_next"] > threshold).astype(int)


In [82]:
split_index = int(len(model_df) * 0.8)
train = model_df.iloc[:split_index]
test  = model_df.iloc[split_index:]

# Regression targets (continuous)
X_train = train[features_updated]
y_train = train[target]  # e.g. "aqi_change"

X_test = test[features_updated]
y_test = test[target]

# Classification targets (binary) — define spike threshold
SPIKE_THRESHOLD = 20  # adjust based on your AQI scale

X_traincl = train[features_updated]
y_traincl = (train[target] > SPIKE_THRESHOLD).astype(int)

X_testcl = test[features_updated]
y_testcl = (test[target] > SPIKE_THRESHOLD).astype(int)

print("Class distribution:\n", y_traincl.value_counts())

Class distribution:
 AQI_change_next
0    12735
1     4044
Name: count, dtype: int64


In [83]:
from lightgbm import LGBMClassifier

clf = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    class_weight="balanced",   # VERY IMPORTANT
    random_state=42
)

clf.fit(X_traincl, y_traincl)

y_prob = clf.predict_proba(X_test)[:, 1]


[LightGBM] [Info] Number of positive: 4044, number of negative: 12735
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001127 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4905
[LightGBM] [Info] Number of data points in the train set: 16779, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [84]:
from sklearn.metrics import classification_report
y_pred = (y_prob > 0.3).astype(int)
print(classification_report(y_testcl, y_pred, digits=3))

              precision    recall  f1-score   support

           0      0.943     0.694     0.800      3327
           1      0.418     0.840     0.558       868

    accuracy                          0.724      4195
   macro avg      0.680     0.767     0.679      4195
weighted avg      0.834     0.724     0.750      4195



In [85]:
# ─── MODEL A: CLASSIFIER BASELINE ─────────────────────────────

clf_A = LGBMClassifier(
    n_estimators=400,
    learning_rate=0.05,
    class_weight='balanced',
    random_state=42
)
clf_A.fit(X_traincl[weather_features], y_traincl)

y_prob_A  = clf_A.predict_proba(X_testcl[weather_features])[:, 1]
y_pred_A_cls = (y_prob_A > 0.3).astype(int)

print("=== MODEL A CLASSIFIER (Weather Only) ===")
print(classification_report(y_testcl, y_pred_A_cls, digits=3))

[LightGBM] [Info] Number of positive: 4044, number of negative: 12735
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.004157 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2865
[LightGBM] [Info] Number of data points in the train set: 16779, number of used features: 12
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
=== MODEL A CLASSIFIER (Weather Only) ===
              precision    recall  f1-score   support

           0      0.938     0.662     0.776      3327
           1      0.391     0.832     0.532       868

    accuracy                          0.697      4195
   macro avg      0.665     0.747     0.654      4195
weighted avg      0.825     0.697     0.726      4195



In [86]:
threshold = df_feat['AQI_change_next'].quantile(0.90)

df_feat['spike'] = (
    df_feat['AQI_change_next'] > threshold
).astype(int)
df_feat['spike'].value_counts(normalize=True)

,proportion
spike,
0,0.899991
1,0.100009


In [90]:
from lightgbm import LGBMClassifier

clf = LGBMClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42
)

clf.fit(X_train, y_traincl)

y_pred_class = clf.predict(X_test)


[LightGBM] [Info] Number of positive: 4044, number of negative: 12735
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.003876 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 4905
[LightGBM] [Info] Number of data points in the train set: 16779, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


In [91]:
# ─── HEAD TO HEAD COMPARISON ──────────────────────────────────

print("=" * 45)
print(f"{'Metric':<12} {'Model A (Weather)':>18} {'Model B (+ Constr)':>18}")
print("=" * 45)
print(f"{'RMSE':<12} {rmse_A:>18.4f} {rmse:>18.4f}")
print(f"{'MAE':<12} {mae_A:>18.4f} {mae:>18.4f}")
print(f"{'R²':<12} {r2_A:>18.4f} {r2:>18.4f}")
print("=" * 45)
print("Lower RMSE/MAE = better  |  Higher R² = better")
print("If Model B wins → H2 supported")
print("If Model A ties → construction proxy adds no signal")

Metric        Model A (Weather) Model B (+ Constr)
RMSE                    31.9439            29.5336
MAE                     20.7826            19.4021
R²                       0.3533             0.4472
Lower RMSE/MAE = better  |  Higher R² = better
If Model B wins → H2 supported
If Model A ties → construction proxy adds no signal


In [92]:
from sklearn.metrics import classification_report

print(classification_report(y_testcl, y_pred_class))


              precision    recall  f1-score   support

           0       0.91      0.85      0.88      3327
           1       0.54      0.66      0.59       868

    accuracy                           0.81      4195
   macro avg       0.72      0.76      0.74      4195
weighted avg       0.83      0.81      0.82      4195



In [ ]:
print("features_updated length:", len(features_updated))
print(features_updated)


In [ ]:
print("features_updated:",
      len(features_updated))
print("reg n_features_:", reg.n_features_in_)
print("clf n_features_:", clf.n_features_in_)

In [ ]:
import joblib

bundle = {
    "reg_model": reg,
    "clf_model": clf,
    "features": features_updated,
    "spike_prob_threshold": 0.30
}

joblib.dump(bundle, "aqi_bundle.pkl")
